[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module4/b_Many_To_Many_BDL_tmpf_and_tmpfPlus1.ipynb)

# b_Many To Many (Numeric Sequences)
---------------------------------
**Dr. Dave Wanik - University of Connecticut**

[y is one variable, but predicting two timesteps into the future - NOT AUTOREGRESSIVE]

Let's read in the BDL data and see if we can predict two or three time steps into the future. We call this n_outputs or time steps into the future.

This script works by using the same LSTM cell, connecting to a dense layer, but the dense layer has hidden_units = n_outputs.

In [1]:
# import modules
from numpy import array
from tensorflow.keras.preprocessing.text import one_hot
#from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from tensorflow.keras.layers import Activation, Dropout, Dense
from keras.layers import Flatten, LSTM
from keras.layers import GlobalMaxPooling1D
from keras.models import Model
#from keras.layers.embeddings import Embedding
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.layers import Input
#from keras.layers.merge import Concatenate
from keras.layers import Bidirectional

import pandas as pd
import numpy as np
import re

import matplotlib.pyplot as plt

In [2]:
# # https://drive.google.com/file/d/1vhWT7__EDc-WQ7WGnK0RP2qq7-25LCWS/view?usp=sharing
# !gdown 1vhWT7__EDc-WQ7WGnK0RP2qq7-25LCWS
# # read the data
# df = pd.read_csv('../data/cleanBDL.csv')

In [3]:
# Link to the data file on Github
url = "https://raw.githubusercontent.com/drdave-teaching/OPIM5509Files/refs/heads/main/OPIM5509_Module4_Files/data/cleanBDL.csv"

# retrieve the CSV data and build a dataframe
df = pd.read_csv(url)

df.shape

(46272, 10)

In [4]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 46272 entries, 0 to 46271
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   valid   46272 non-null  str    
 1   tmpf    46272 non-null  float64
 2   dwpf    46272 non-null  float64
 3   relh    46272 non-null  float64
 4   drct    46272 non-null  float64
 5   sknt    46272 non-null  float64
 6   p01i    46272 non-null  float64
 7   alti    46272 non-null  float64
 8   mslp    46272 non-null  float64
 9   vsby    46272 non-null  float64
dtypes: float64(9), str(1)
memory usage: 4.4 MB


,valid,tmpf,dwpf,relh,drct,sknt,p01i,alti,mslp,vsby
0,2015-01-01 00:00:00,17.96,6.08,59.10,190.0,5.0,0.0,30.09,1019.0,10.0
1,2015-01-01 01:00:00,19.94,8.06,59.40,190.0,5.0,0.0,30.08,1018.7,10.0
2,2015-01-01 02:00:00,23.00,6.98,49.69,210.0,9.0,0.0,30.06,1018.1,10.0
3,2015-01-01 03:00:00,21.92,5.00,47.52,230.0,11.0,0.0,30.04,1017.4,10.0
4,2015-01-01 04:00:00,23.00,3.92,43.21,250.0,13.0,0.0,30.05,1017.7,10.0


# Define X and Y
If we are going to use our split sequences script from Brownlee, then we need to make sure our Y variables are on the end!

In [5]:
# let's drop the valid column
# Y will be dwpf and relh
# X will be everything else!

del df['valid']
df.head() # check your work

,tmpf,dwpf,relh,drct,sknt,p01i,alti,mslp,vsby
0,17.96,6.08,59.10,190.0,5.0,0.0,30.09,1019.0,10.0
1,19.94,8.06,59.40,190.0,5.0,0.0,30.08,1018.7,10.0
2,23.00,6.98,49.69,210.0,9.0,0.0,30.06,1018.1,10.0
3,21.92,5.00,47.52,230.0,11.0,0.0,30.04,1017.4,10.0
4,23.00,3.92,43.21,250.0,13.0,0.0,30.05,1017.7,10.0


In [6]:
Y = df[['tmpf']]
X = df.drop(columns=['tmpf'])
print(df.shape, X.shape, Y.shape)

# looks good! Let's prepare samples for modeling

(46272, 9) (46272, 8) (46272, 1)


In [7]:
# put Y all the way on the left
df = pd.concat([X, Y], axis=1, sort=False)
df.head(n=11)

,dwpf,relh,drct,sknt,p01i,alti,mslp,vsby,tmpf
0,6.08,59.10,190.0,5.0,0.0,30.09,1019.0,10.0,17.96
1,8.06,59.40,190.0,5.0,0.0,30.08,1018.7,10.0,19.94
2,6.98,49.69,210.0,9.0,0.0,30.06,1018.1,10.0,23.00
3,5.00,47.52,230.0,11.0,0.0,30.04,1017.4,10.0,21.92
4,3.92,43.21,250.0,13.0,0.0,30.05,1017.7,10.0,23.00
5,3.92,43.21,250.0,11.0,0.0,30.06,1018.1,10.0,23.00
6,3.02,41.45,240.0,13.0,0.0,30.07,1018.4,10.0,23.00
7,3.92,41.30,240.0,11.0,0.0,30.08,1018.9,10.0,24.08
8,3.92,38.03,210.0,8.0,0.0,30.08,1018.9,10.0,26.06
9,3.92,35.05,220.0,14.0,0.0,30.08,1018.8,10.0,28.04


In [8]:
# some eda - we should be able to predict temperature!
df.plot.scatter(x='tmpf', y='dwpf')
plt.show()

C:\Users\dww05002\AppData\Local\Temp\ipykernel_41684\2614875674.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# to get our other code to run, we will put Y
# on the end then re-run our code (needs updating from blog)

# prep data for modeling (multivariate)
# link: https://machinelearningmastery.com/how-to-develop-lstm-models-for-time-series-forecasting/

from numpy import array

# split a multivariate sequence into samples
def split_sequences(sequences, n_steps):
	X, y = list(), list()
	for i in np.arange(len(sequences)): # be careful of this line!
		# find the end of this pattern
		end_ix = i + n_steps
		# check if we are beyond the dataset
		if end_ix > len(sequences):
			break
		# gather input and output parts of the pattern
    # X and Y have been UPDATED so the last two columns drop off
		seq_x, seq_y = sequences[i:end_ix, :-1], sequences[end_ix-1, -1:]
		X.append(seq_x)
		y.append(seq_y)
	return np.array(X), np.array(y)

# Prepare Samples for Modeling
Everything needs to be in 3D arrays.

In [10]:
# let's turn X into lookbacks of 10 with all of our samples
# all we need to do is decide on is n_steps (what our lookback period is)
# since we have a bunch of data, why not n_steps=10? then try 30 later on.
n_steps = 10
raw_seq = np.array(df) #make sure your data is stored as a numpy array!
# let's ignore the date column and just use the temperature data
X, y = split_sequences(raw_seq, n_steps=10)

In [11]:
# check your work
print(df.shape, X.shape, y.shape)

(46272, 9) (46263, 10, 8) (46263, 1)


In [12]:
# here's the first X
X[0]

array([[   6.08,   59.1 ,  190.  ,    5.  ,    0.  ,   30.09, 1019.  ,
          10.  ],
       [   8.06,   59.4 ,  190.  ,    5.  ,    0.  ,   30.08, 1018.7 ,
          10.  ],
       [   6.98,   49.69,  210.  ,    9.  ,    0.  ,   30.06, 1018.1 ,
          10.  ],
       [   5.  ,   47.52,  230.  ,   11.  ,    0.  ,   30.04, 1017.4 ,
          10.  ],
       [   3.92,   43.21,  250.  ,   13.  ,    0.  ,   30.05, 1017.7 ,
          10.  ],
       [   3.92,   43.21,  250.  ,   11.  ,    0.  ,   30.06, 1018.1 ,
          10.  ],
       [   3.02,   41.45,  240.  ,   13.  ,    0.  ,   30.07, 1018.4 ,
          10.  ],
       [   3.92,   41.3 ,  240.  ,   11.  ,    0.  ,   30.08, 1018.9 ,
          10.  ],
       [   3.92,   38.03,  210.  ,    8.  ,    0.  ,   30.08, 1018.9 ,
          10.  ],
       [   3.92,   35.05,  220.  ,   14.  ,    0.  ,   30.08, 1018.8 ,
          10.  ]])

In [13]:
# here's the first Y
y[0]

# go scroll up and make sure this matches!
# and it does!

# you will need to customize your split script when
# prepping your data... be careful! take control of your data!

array([28.04])

## Y, Y+1 and Y+2
Let's make sure that our Y vector has multiple time steps, just like we did in the previous script.

In [14]:
# convert to a dataframe
tmp = pd.DataFrame(y)
# rename the column
tmp.rename(columns={0:'y'}, inplace=True)
# create some shifts
tmp['yPlus1'] = tmp['y'].shift(-1)
tmp['yPlus2'] = tmp['y'].shift(-2)
# check your work
print(tmp.head())
print(tmp.tail()) # we will have to deal with those NaN's

# either ffill them or delete them later.

       y  yPlus1  yPlus2
0  28.04   30.02   32.00
1  30.02   32.00   33.08
2  32.00   33.08   33.98
3  33.08   33.98   33.08
4  33.98   33.08   33.08
          y  yPlus1  yPlus2
46258  44.1    42.1    39.0
46259  42.1    39.0    39.9
46260  39.0    39.9    37.0
46261  39.9    37.0     NaN
46262  37.0     NaN     NaN


In [15]:
# I will opt to forward fill them
# and just except the dirt in my data
tmp = tmp.ffill()   # pandas 3 removed fillna(method=)
tmp.tail() # all better!

# of course, you could have dropped the last N arrays
# from both X and y (making sure the shape matches up)

,y,yPlus1,yPlus2
46258,44.1,42.1,39.0
46259,42.1,39.0,39.9
46260,39.0,39.9,37.0
46261,39.9,37.0,37.0
46262,37.0,37.0,37.0


In [16]:
# convert back to numpy array
y = tmp

In [17]:
y.shape

(46263, 3)

# Fit a Model
This will be similar to the last example in 'Sequence Problems_Pt1.ipynb'

In [18]:
# note how there's a 2 at the end
# usually we did this for a multi-classification problem, but not today!
# by default, it's a 'linear' activiation function
# so this is 2 node output and we're doing regression.

n_steps = X.shape[1]
n_features = X.shape[2]
n_outputs = y.shape[1]

print(n_steps, n_features, n_outputs)

10 8 3


In [19]:
model = Sequential()
model.add(LSTM(50, activation='relu',
               recurrent_dropout=0.1,
               input_shape=(n_steps, n_features)))
model.add(Dropout(0.2))
model.add(Dense(n_outputs)) # since Y has three values, we need to predict three values
model.compile(optimizer='adam', loss='mse')

import keras
from keras.callbacks import EarlyStopping

# early stopping callback
# This callback will stop the training when there is no improvement in
# the validation loss for 10 consecutive epochs.
es = keras.callbacks.EarlyStopping(monitor='val_loss',
                                   mode='min',
                                   patience=10, # you can play with this!
                                   restore_best_weights=True) # important - otherwise you just return the last weigths...

# now we just update our model fit call
history = model.fit(X,
                    y,
                    callbacks=[es],
                    epochs=800, # you can set this to a big number!
                    batch_size=100,
                    validation_split=0.2,
                    verbose=1,
                    shuffle=True)

Epoch 1/800


C:\Users\dww05002\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  1/371 ━━━━━━━━━━━━━━━━━━━━ 14:27 2s/step - loss: 23076.3496

 11/371 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 18480.3965  

 21/371 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 14539.0801

...

340/371 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 78.0889

349/371 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 77.9376

358/371 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 78.0717

367/371 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 78.0863

371/371 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 78.0868 - val_loss: 11.9137


In [20]:
# and you can show how your model did

# since it's a single sample, we need to reshape
data = X[0]
data = data.reshape(1,n_steps,n_features)
print(data)
print(model.predict(data))

# did we get close?
print(y[0:1])

# of course you can show scatterplots and everything else
# for more examples

[[[   6.08   59.1   190.      5.      0.     30.09 1019.     10.  ]
  [   8.06   59.4   190.      5.      0.     30.08 1018.7    10.  ]
  [   6.98   49.69  210.      9.      0.     30.06 1018.1    10.  ]
  [   5.     47.52  230.     11.      0.     30.04 1017.4    10.  ]
  [   3.92   43.21  250.     13.      0.     30.05 1017.7    10.  ]
  [   3.92   43.21  250.     11.      0.     30.06 1018.1    10.  ]
  [   3.02   41.45  240.     13.      0.     30.07 1018.4    10.  ]
  [   3.92   41.3   240.     11.      0.     30.08 1018.9    10.  ]
  [   3.92   38.03  210.      8.      0.     30.08 1018.9    10.  ]
  [   3.92   35.05  220.     14.      0.     30.08 1018.8    10.  ]]]


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 321ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 343ms/step


[[27.32622  26.807281 26.402674]]
       y  yPlus1  yPlus2
0  28.04   30.02    32.0


In [21]:
# select first column of y
y['y']

0        28.04
1        30.02
2        32.00
3        33.08
4        33.98
         ...  
46258    44.10
46259    42.10
46260    39.00
46261    39.90
46262    37.00
Name: y, Length: 46263, dtype: float64

In [22]:
# well done! You can also make scatterplots of actual vs. predicted
pred = model.predict(X)
pred

   1/1446 ━━━━━━━━━━━━━━━━━━━━ 6:54 287ms/step

  30/1446 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step    

  60/1446 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step

  88/1446 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step

 118/1446 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step

...

1368/1446 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

1395/1446 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

1422/1446 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

1446/1446 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step


array([[27.32622 , 26.807281, 26.402674],
       [29.13072 , 28.60801 , 28.162449],
       [30.303923, 29.740906, 29.22007 ],
       ...,
       [37.59651 , 36.94209 , 36.358448],
       [40.236202, 39.61105 , 38.980526],
       [35.6823  , 35.06623 , 34.59759 ]], shape=(46263, 3), dtype=float32)

In [23]:
# select first column
pred[:,0:1]

array([[27.32622 ],
       [29.13072 ],
       [30.303923],
       ...,
       [37.59651 ],
       [40.236202],
       [35.6823  ]], shape=(46263, 1), dtype=float32)

In [24]:
# pred1
plt.scatter(y['y'], pred[:,0:1])
plt.show()

C:\Users\dww05002\AppData\Local\Temp\ipykernel_41684\1016567324.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [25]:
# pred2
plt.scatter(y['yPlus1'], pred[:,1:2])
plt.show()

C:\Users\dww05002\AppData\Local\Temp\ipykernel_41684\3354951425.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [26]:
# and the third one
plt.scatter(y['yPlus2'], pred[:,1:2])
plt.show()

C:\Users\dww05002\AppData\Local\Temp\ipykernel_41684\2471302604.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [27]:
# it worked! of course the prediction into the
# future will be a little worst than the next time step

# and as you can see, prepping the data is important!